# Employee Turnover Prediction
## Internship project: Decision Tree vs Random Forest

**Target:** `quit` (1 = left, 0 = stayed). This notebook uses the provided dataset and a reproducible stratified 80/20 split. Evaluate recall/F1 for the `quit` class as well as accuracy because the classes are imbalanced.

## 1. Load libraries and dataset

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, classification_report,
                             ConfusionMatrixDisplay)

DATA_PATH = Path("data/employee_data.csv")
if not DATA_PATH.exists():
    DATA_PATH = Path("../data/employee_data.csv")
df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
display(df.head())
display(df.isna().sum())

## 2. Explore the target and features

In [ ]:
display(df.describe(include="all").T)
display(df["quit"].value_counts().rename(index={0: "Stayed", 1: "Quit"}))

df["quit"].value_counts().sort_index().plot(kind="bar", title="Target class counts")
plt.xlabel("quit (0 = stayed, 1 = left)")
plt.ylabel("Employees")
plt.tight_layout()
plt.show()

pd.crosstab(df["salary"], df["quit"], normalize="index").plot(kind="bar", stacked=True)
plt.title("Turnover proportion by salary category")
plt.ylabel("Proportion")
plt.tight_layout()
plt.show()

## 3. Prepare features and split data

In [ ]:
target = "quit"
numeric_features = [
    "satisfaction_level", "last_evaluation", "number_project",
    "average_montly_hours", "time_spend_company", "Work_accident",
    "promotion_last_5years"
]
categorical_features = ["department", "salary"]
X = df[numeric_features + categorical_features]
y = df[target].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print("Train:", X_train.shape, "Test:", X_test.shape)
print("Train class proportions:\n", y_train.value_counts(normalize=True))

## 4. Build preprocessing pipeline and compare models

In [ ]:
numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])
preprocess = ColumnTransformer([
    ("numeric", numeric_pipe, numeric_features),
    ("categorical", categorical_pipe, categorical_features)
])

models = {
    "Decision Tree": DecisionTreeClassifier(
        max_depth=6, min_samples_leaf=10, class_weight="balanced", random_state=42
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=250, min_samples_leaf=3, class_weight="balanced",
        random_state=42, n_jobs=-1
    )
}
fitted_models = {}
results = []
for name, classifier in models.items():
    pipe = Pipeline([("preprocess", preprocess), ("model", classifier)])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    proba = pipe.predict_proba(X_test)[:, 1]
    fitted_models[name] = pipe
    results.append({
        "model": name,
        "accuracy": accuracy_score(y_test, pred),
        "precision_quit": precision_score(y_test, pred, zero_division=0),
        "recall_quit": recall_score(y_test, pred, zero_division=0),
        "f1_quit": f1_score(y_test, pred, zero_division=0),
        "roc_auc": roc_auc_score(y_test, proba)
    })
    print("\n", name)
    print(classification_report(y_test, pred, target_names=["Stayed", "Quit"], zero_division=0))
    ConfusionMatrixDisplay.from_predictions(
        y_test, pred, display_labels=["Stayed", "Quit"], colorbar=False
    )
    plt.title(f"{name} — Confusion Matrix")
    plt.tight_layout()
    plt.show()

results_df = pd.DataFrame(results).sort_values("f1_quit", ascending=False)
display(results_df)

## 5. Feature importance (Random Forest)

In [ ]:
rf_pipe = fitted_models["Random Forest"]
feature_names = rf_pipe.named_steps["preprocess"].get_feature_names_out()
importances = rf_pipe.named_steps["model"].feature_importances_
importance_df = pd.DataFrame({"feature": feature_names, "importance": importances})
importance_df = importance_df.sort_values("importance", ascending=False).head(15)
display(importance_df)

importance_df.sort_values("importance").plot(
    x="feature", y="importance", kind="barh", legend=False, figsize=(9, 6)
)
plt.title("Top 15 Random Forest feature importances")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()

## 6. Conclusions and limitations

- Compare recall and F1 for employees who left, not only overall accuracy.
- Feature importance indicates model reliance, not causation.
- The dataset may not represent a real organization or current workforce.
- Do not use the output as the sole basis for decisions about employees; check fairness, privacy and data quality.
- The accompanying `app.py` provides an interactive demonstration after training with `python train_model.py`.